[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# Introducción a CI/CD y *GitHub Actions*.

Este cuaderno define los tres modelos de entrega continua, presenta la anatomía de [*GitHub Actions*](https://docs.github.com/actions) y muestra un workflow completo antes de entrar a la sintaxis en detalle.

## Objetivos.

- Distinguir integración continua, entrega continua y despliegue continuo.
- Entender qué es un runner, un job, un step y una action.
- Leer y escribir un workflow mínimo funcional.
- Preparar la base para quality gates, contenedores, IaC y despliegue seguro.

## Integración continua, entrega continua y despliegue continuo.

Los tres términos describen compromisos distintos sobre qué tan lejos llega la automatización después de un `git push`.

**Integración continua (CI)** significa que cada cambio que llega al repositorio dispara automáticamente una construcción y una suite de pruebas. La idea es simple: cuanto antes se detecta un conflicto o un error, más barato es corregirlo. Un push que rompe tests debe fallar en minutos, no en días cuando alguien intenta hacer un release.

**Entrega continua (Continuous Delivery)** extiende CI: además de verificar, el pipeline produce un **artefacto** —una imagen Docker, un paquete, un binario— que en cualquier momento está listo para desplegarse a producción. El despliegue sigue requiriendo que alguien apriete un botón, pero ese botón siempre está disponible. No hay que "preparar un release": si el pipeline pasa, hay algo desplegable.

**Despliegue continuo (Continuous Deployment)** elimina ese botón: si el pipeline completo pasa, el artefacto llega a producción de forma automática. Es el modelo de mayor velocidad y el que más exige: sin buena cobertura de pruebas, observabilidad y capacidad de rollback, un bug puede llegar a producción antes de que alguien lo note.

| Práctica | Build + test automático | Artefacto siempre desplegable | Despliegue automático a producción |
|---|---|---|---|
| Integración continua | ✓ | — | — |
| Entrega continua | ✓ | ✓ | — |
| Despliegue continuo | ✓ | ✓ | ✓ |

La sigla **CI/CD** es ambigua: puede referirse a integración continua + entrega continua o a integración continua + despliegue continuo. En la práctica, la mayoría de los equipos opera en modo de entrega continua —con una aprobación explícita antes de producción— y usa "CI/CD" para describir ese modelo.

Desde la perspectiva de seguridad, los tres modelos comparten el mismo problema: el pipeline tiene acceso privilegiado a secretos, registros de imágenes y entornos de producción. La diferencia está en quién autoriza el uso de ese acceso: una persona en entrega continua, el pipeline mismo en despliegue continuo.

### Promoción entre entornos.

En la práctica, "producción" no es el único destino. El artefacto que el pipeline construyó y probó recorre una secuencia de entornos antes de llegar a los usuarios: típicamente `dev`, `staging` y `prod`. Cada entorno tiene sus propios controles: variables de configuración distintas, secretos acotados, aprobaciones requeridas. A este recorrido se le llama **promoción entre entornos**.

El principio es que se promueve el mismo artefacto, no se reconstruye uno nuevo por entorno. Lo que cambia entre entornos son las políticas, no el código. Reconstruir introduce la posibilidad de que lo que se probó no sea exactamente lo que se desplegó.

Este mecanismo convierte a CI/CD de una herramienta de automatización en un sistema de control: cada entorno es un punto de verificación antes de que el artefacto llegue a producción.

## La anatomía de *GitHub Actions*.

Durante el ciclo de vida del software aparecen tareas repetitivas y susceptibles de error humano: correr tests, verificar formato, construir una imagen, desplegar. *GitHub Actions* convierte esas tareas en automatizaciones que se disparan ante eventos del repositorio.

La unidad mínima es el **workflow**: un archivo YAML en `.github/workflows/` que define cuándo y qué ejecutar. Dentro de un workflow:

- **Job**: unidad de trabajo que corre en su propio entorno. Los jobs de un workflow corren en paralelo por defecto.
- **Step**: paso individual dentro de un job. Los steps de un job corren en secuencia, en el mismo entorno.
- **Action**: step empaquetado y reutilizable. Puede ser oficial de *GitHub* (`actions/checkout`), de la comunidad (GitHub Marketplace) o propia del repositorio.
- **Runner**: máquina efímera donde corre el job. *GitHub* provisiona y destruye el runner por cada ejecución; lo que se instala en un job no persiste al siguiente.

```text
workflow  (.github/workflows/ci.yml)
└── job: checks  (corre en ubuntu-latest)
    ├── step: actions/checkout@v4       ← action oficial
    ├── step: actions/setup-python@v5   ← action oficial
    ├── step: run: ruff check .         ← comando shell
    └── step: run: pytest               ← comando shell
```

Herramientas nuevas en este bloque: [actions/checkout](https://github.com/actions/checkout), [actions/setup-python](https://github.com/actions/setup-python), [*ruff*](https://docs.astral.sh/ruff/) y [*pytest*](https://docs.pytest.org/).

<img src="img/05/github_actions_basico.svg" width="70%">

## Mini tutorial visual: primer run en GitHub Actions.

Este anexo muestra el recorrido mínimo en la interfaz para crear y ejecutar un workflow.

1. Ir a la pestaña **Actions** del repositorio.

<img src="img/05/05-actions-01-tab-actions.png" width="85%">

2. Crear el archivo de workflow inicial desde Actions.

<img src="img/05/05-actions-02-crear-workflow.png" width="85%">

3. Ejecutar el workflow manualmente con **Run workflow**.

<img src="img/05/05-actions-03-run-workflow.png" width="85%">

4. Verificar estado exitoso del run en el resumen.

<img src="img/05/05-actions-04-resumen-exito.png" width="85%">

5. Revisar logs del job para validar que el pipeline corrió como se esperaba.

<img src="img/05/05-actions-05-logs-job.png" width="85%">

Este bloque es de apoyo visual: la referencia principal sigue siendo el YAML del workflow y la lectura de logs.

## Un workflow completo mínimo.

El siguiente workflow implementa un pipeline de CI básico: se dispara en cada push y pull request, instala dependencias, corre el linter y ejecuta los tests.

```yaml
# .github/workflows/ci.yml
name: ci

on:
  push:
    branches: ["main"]
  pull_request:

jobs:
  checks:
    runs-on: ubuntu-latest        # runner efímero provisto por *GitHub*

    steps:
      - uses: actions/checkout@v4             # clona el repositorio
      - uses: actions/setup-python@v5         # instala *Python* en el runner
        with:
          python-version: "3.12"

      - run: pip install -r requirements.txt  # comando shell directo
      - run: ruff check .                     # linter
      - run: pytest                           # tests
```

Tres cosas que vale la pena notar:

- `uses:` referencia una action por repositorio y tag. El `@v4` fija la versión; en capítulos posteriores se verá por qué anclar por SHA es más seguro que anclar por tag.
- `run:` ejecuta cualquier comando de shell disponible en el runner. El runner es Ubuntu, así que todo lo que funciona en bash funciona aquí.
- Si cualquier step falla, el job falla y *GitHub* marca el check como rojo. Mientras el check esté rojo, una rama protegida no permite hacer merge.

Este workflow es el punto de partida. El cuaderno siguiente cubre en detalle cada parte de la sintaxis: triggers, contextos, dependencias entre jobs, condicionales y control de ejecución.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>